In [1]:
from workflow_auto_assembler import WorkflowAutoAssembler, AssembledWorkflow, create_avc_items, LlmFunctionItemInput

### 1. Define available tools

In [2]:
from typing import Type, List
from pydantic import BaseModel, Field

# --- Example usage ---

# Define mock functions and their associated Pydantic models:

# 1. get_weather function

class GetWeatherInput(BaseModel):
    city: str = Field(..., description="Name of the city for which weather to be extracted.")

class GetWeatherOutput(BaseModel):
    condition: str = Field(..., description="Weather condition in the requested city.")
    temperature: float = Field(..., description="Termperature in C in the requested city.")
    humidity: float = Field(None, description="Name of the city for which weather to be extracted.")

def get_weather(inputs: GetWeatherInput) -> GetWeatherOutput:
    """Get the current weather for a city from weather forcast api."""
    return GetWeatherOutput(
        condition = "Sunny",
        temperature = 20,
        humidity = 0.6
    )

# 2. send_report_email function

class EmailInformationPoint(BaseModel):
    title: str = Field(None, description="Few word description of the information.")
    content: str = Field(..., description="Content of the information.")

class SendReportEmailInput(BaseModel):
    city: str = Field(..., description="Name of the city where report will be send.")
    information: list[EmailInformationPoint]

class SendReportEmailOutput(BaseModel):
    email_sent: bool = Field(..., description="Conformation that email was send successfully.")
    message: str = Field(None, description="Optional comments from the process.")

def send_report_email(inputs: SendReportEmailInput) -> SendReportEmailOutput:
    """Sends a report email with given information points to a city."""
    return SendReportEmailOutput(
        email_sent = True,
        message = "Email sent to city of your choosing!"
    )

# 3. query_database function

class QueryDatabaseInput(BaseModel):
    topic: str = Field(..., description="Topic of a requested piece of information.")
    location: str = Field(None, description="Filter for location name.")
    uid: str = Field(None, description="Filter for unique indentifier of the database item.")

class QueryDatabaseOutput(BaseModel):
    info: str = Field(..., description="Content of the information.")
    uid: str = Field(None, description="Unique indentifier of the database item.")

def query_database(inputs : QueryDatabaseInput) -> QueryDatabaseOutput:
    """Get information from the database with provided filters."""
    return QueryDatabaseOutput(
        info = "Content extracted from the database for your query is ...",
        uid = "0000"
    )

# 4. query_web function

class QueryWebInput(BaseModel):
    search_input: str = Field(..., description="Topic to be searched on the web.")


class QueryWebOutput(BaseModel):
    search_results: List[str] = Field(..., description="List relevant info from search results.")


def query_web(inputs : QueryWebInput) -> QueryWebOutput:
    """Get information from the internet for provided query."""
    return QueryWebOutput(
        search_results = ["Relevant content found in first search result is ..."],
    )



# Create structured items for each function

available_tools = create_avc_items(functions = [
    LlmFunctionItemInput(**{"func" : get_weather , "input_model" : GetWeatherInput, "output_model" : GetWeatherOutput}),
    LlmFunctionItemInput(**{"func" : send_report_email , "input_model" : SendReportEmailInput, "output_model" : SendReportEmailOutput}),
    LlmFunctionItemInput(**{"func" : query_database , "input_model" : QueryDatabaseInput, "output_model" : QueryDatabaseOutput}),
    LlmFunctionItemInput(**{"func" : query_web , "input_model" : QueryWebInput, "output_model" : QueryWebOutput})
])

### 2. Define task, expected inputs and outputs

In [3]:
task_description = """Query database to find information on birds and get latest weather for the city, then send an email there."""

class WfInputs(BaseModel):
    city: str = Field(..., description="Name of the city for which weather to be extracted.")

class WfOutputs(BaseModel):
    city: str = Field(..., description="Name of the city for which weather was extracted.")
    information: list[EmailInformationPoint] = Field(default=[], description="Information sent via email.")

### 3. Initialize and run workflow assembler

In [4]:
AssembledWorkflow.model_fields

{'id': FieldInfo(annotation=str, required=True, description='Unique id for task based on hash of inputs'),
 'input_id': FieldInfo(annotation=str, required=True, description='Unique id for task inputs based on hash of inputs'),
 'init_check': FieldInfo(annotation=Union[WorkflowCheckResponse, NoneType], required=False, default=None, description='Initial check.'),
 'planning': FieldInfo(annotation=Union[PlanningStepsResp, NoneType], required=False, default=PlanningStepsResp(planner=None, planner_iters=[], adaptor=None, adaptor_iters=[], tester=None, planner_rerun_needed=True, adaptor_rerun_needed=True, testing_errors=[], test_retries=0), description='Responses from planning steps.'),
 'workflow_possible': FieldInfo(annotation=Union[bool, NoneType], required=False, default=None, description='Indicates if workflow could be planned given provided tools.'),
 'workflow_completed': FieldInfo(annotation=Union[bool, NoneType], required=False, default=False, description='Indicates if workflow was 

In [5]:
import logging

wa = WorkflowAutoAssembler(
    available_functions = available_tools["available_functions"],
    available_callables = available_tools["available_callables"],
    llm_handler_params = {
        "llm_h_type" : "ollama",
        "llm_h_params" : {
            "connection_string": "http://localhost:11434",
            "model_name": "gpt-oss:20b"
        }
    },
    loggerLvl=logging.DEBUG
)

wf_obj = await wa.plan_workflow(
    task_description = task_description,
    input_model = WfInputs,
    output_model = WfOutputs,
    test_params = [
        {
            "inputs" : WfInputs(city = "Berlin"), 
            "outputs" : WfOutputs(
                city='Berlin', 
                information=[
                    EmailInformationPoint(title='Birds Info', content='Content extracted from the database for your query is ...'), 
                    EmailInformationPoint(title='Weather', content='Sunny')])},
        {
            "inputs" : WfInputs(city = "Paris"), 
            "outputs" : WfOutputs(
                city='Paris', 
                information=[
                    EmailInformationPoint(title='Birds Info', content='Content extracted from the database for your query is ...'), 
                    EmailInformationPoint(title='Weather', content='Sunny')])}
    ],

    #compare_params={"ignore_fields" : ["information"]}
)

DEBUG:WorkflowAutoAssembler:Starting workflow planning ...
DEBUG:WorkflowAutoAssembler:Running initial check...
DEBUG:WorkflowAutoAssembler:-> 1
DEBUG:WorkflowAutoAssembler:-> 2
DEBUG:WorkflowAutoAssembler:-> 3
DEBUG:WorkflowAutoAssembler:-> 4
DEBUG:WorkflowAutoAssembler:-> 5
DEBUG:WorkflowAutoAssembler:-> 6
DEBUG:WorkflowAutoAssembler:-> 7
DEBUG:WorkflowAutoAssembler:<- 4
DEBUG:WorkflowAutoAssembler:<- 6
DEBUG:WorkflowAutoAssembler:<- 5
DEBUG:WorkflowAutoAssembler:<- 1
DEBUG:WorkflowAutoAssembler:<- 7
DEBUG:WorkflowAutoAssembler:<- 3
DEBUG:WorkflowAutoAssembler:<- 2
DEBUG:WorkflowAutoAssembler:Decision from initial check is ready!
DEBUG:WorkflowAutoAssembler:-> 8
DEBUG:WorkflowAutoAssembler:<- 8
DEBUG:WorkflowAutoAssembler:Adapting all steps!
DEBUG:WorkflowAutoAssembler:-> 9
DEBUG:WorkflowAutoAssembler:-> 10
DEBUG:WorkflowAutoAssembler:-> 11
DEBUG:WorkflowAutoAssembler:-> 12
DEBUG:WorkflowAutoAssembler:<- 9
DEBUG:WorkflowAutoAssembler:Reset caused by ADAPTOR_JSON type error!
DEBUG:Wor

In [12]:
wf_obj.id

'4b19095a61944997a8c90f1d58f21ae2'

In [13]:
wf_obj.workflow_possible

True

In [14]:
wf_obj.workflow_completed

True

In [15]:
wf_obj.planning.test_retries

1

In [21]:
wf_obj.planning.tester.case_results

[TestedWorkflow(workflow=[WorkflowItem(name='query_database', args={'topic': 'birds', 'location': '0.output.city'}), WorkflowItem(name='get_weather', args={'city': '0.output.city'}), WorkflowItem(name='send_report_email', args={'city': '0.output.city', 'information': [{'title': 'Birds Information', 'content': '1.output.info'}, {'title': 'Weather Condition', 'content': '2.output.condition'}]}), WorkflowItem(name='output_model', args={'city': '0.output.city', 'information': [{'title': 'Birds Info', 'content': '1.output.info'}, {'title': 'Weather', 'content': '2.output.condition'}]})], inputs=WfInputs(city='Berlin'), outputs={'0': WfInputs(city='Berlin'), '1': QueryDatabaseOutput(info='Content extracted from the database for your query is ...', uid='0000'), '2': GetWeatherOutput(condition='Sunny', temperature=20.0, humidity=0.6), '3': SendReportEmailOutput(email_sent=True, message='Email sent to city of your choosing!'), '4': WfOutputs(city='Berlin', information=[EmailInformationPoint(tit

### 4. Run assembled workflow

In [22]:
wa = WorkflowAutoAssembler(
    available_functions = available_tools["available_functions"],
    available_callables = available_tools["available_callables"],
    llm_handler_params = {
        "llm_h_type" : "ollama",
        "llm_h_params" : {
            "connection_string": "http://localhost:11434",
            "model_name": "gpt-oss:20b"
        }
    },
    loggerLvl=logging.DEBUG
)

output = await wa.run_workflow(
    workflow_object = wf_obj,
    run_inputs = WfInputs(city = "London")
)

output.model_dump()

DEBUG:WorkflowAutoAssembler:Workflow completed after 2 planning loops!


{'city': 'London',
 'information': [{'title': 'Birds Info',
   'content': 'Content extracted from the database for your query is ...'},
  {'title': 'Weather', 'content': 'Sunny'}]}